# Preconditioned Crank-Nicolson (pCN)

Random-walk MH and Langevin proposals both suffer dimension-dependent acceptance rates: as $d \to \infty$ the step size must shrink as $d^{-1/2}$ (RW) or $d^{-1/3}$ (MALA) to keep the acceptance bounded away from zero. For Gaussian-prior posteriors the preconditioned Crank-Nicolson (pCN) proposal

$$
y = \sqrt{1 - \beta^2}\, x + \beta\, L\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I), \quad L L^\top = C,
$$

is *prior-reversible*: when paired with an MH ratio that uses only the **likelihood**, acceptance is dimension-independent.

This notebook demonstrates that property on a synthetic linear-Gaussian regression problem in increasing dimensions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import torch
import matplotlib.pyplot as plt

from mcmc_samplers import (
    MetropolisHastings,
    PreconditionedCrankNicolsonSampler,
    SampleVisualizer,
    diagnostics,
)

## A pCN sampler

pCN is just an MH algorithm with the pCN proposal and a target equal to the *likelihood* (the prior is encoded in the proposal kernel). The package ships a convenience sampler that wires this up.

## Sweep dimension and measure acceptance

We define a synthetic likelihood $\log L(x) = -\frac{1}{2}\|A x - b\|^2$ for a random thin matrix $A$, with a unit-covariance prior. We then run pCN and a tuned random-walk MH for several dimensions and report acceptance.

In [ ]:
def build_problem(dim, seed=0):
    g = torch.Generator().manual_seed(seed)
    A = torch.randn(dim, dim, generator=g) / dim ** 0.5
    x_true = torch.randn(dim, generator=g)
    b = A @ x_true + 0.1 * torch.randn(dim, generator=g)
    def loglik(x):
        r = A @ x - b
        return -0.5 * (r * r).sum()
    return loglik

dims = [4, 16, 64, 256]
results = []
N = 2000
for d in dims:
    lik = build_problem(d)
    pcn = PreconditionedCrankNicolsonSampler(
        likelihood=lik, x0=torch.zeros(d), prior_cov=torch.eye(d), beta=0.3,
        generator=torch.Generator().manual_seed(0),
    )
    samples_pcn, _ = pcn(N, burn_in=N // 5, show_progress=False)
    log_target = lambda x, lik=lik: lik(x) - 0.5 * (x * x).sum()  # full posterior log density
    mh = MetropolisHastings(target=log_target, x0=torch.zeros(d), cov=torch.eye(d) * 0.1 / d, generator=torch.Generator().manual_seed(0))
    samples_mh, _ = mh(N, burn_in=N // 5, show_progress=False)
    results.append((d, pcn.acceptance_ratio, mh.acceptance_ratio))
    print(f'd={d:3d}  pCN acc: {100*pcn.acceptance_ratio:5.1f}%   MH acc: {100*mh.acceptance_ratio:5.1f}%')

In [ ]:
dims_arr = [r[0] for r in results]
fig, ax = plt.subplots()
ax.plot(dims_arr, [r[1] for r in results], 'o-', label='pCN (fixed beta)')
ax.plot(dims_arr, [r[2] for r in results], 's-', label='MH (cov scaled as 1/d)')
ax.set_xscale('log')
ax.set_xlabel('Dimension')
ax.set_ylabel('Acceptance')
ax.legend()
ax.set_title('pCN acceptance is dimension-robust')
plt.show()